In [1]:

# ============================================================
# ✅ STEP 1: LIBRARIES IMPORT
# ============================================================

import yfinance as yf   # market se real option data lene ke liye
import pandas as pd     # data handle aur clean karne ke liye
import numpy as np      # mathematical calculations ke liye (log, exp etc.)
from datetime import datetime  # time to maturity calculate karne ke liye
from scipy.optimize import minimize  # optimization (calibration) ke liye


# ============================================================
# ✅ STEP 2: ASSET SELECT
# ============================================================

ticker = yf.Ticker("AAPL")   # AAPL isliye use kiya kyunki highly liquid hai aur options reliable milte hain


# ============================================================
# ✅ STEP 3: SPOT PRICE
# ============================================================

spot = ticker.history(period="1d")["Close"].iloc[-1]   # current underlying price (S0)
print("Spot:", spot)   # Heston model me S0 ek main input hota hai


# ============================================================
# ✅ STEP 4: EXPIRY FETCH
# ============================================================

expiries = ticker.options   # available option expiry dates
if len(expiries) == 0:
    raise Exception("No options data")   # agar expiry hi nahi mile to code stop

today = datetime.today()
all_data = []   # sab expiries ka data store karne ke liye


# ============================================================
# ✅ STEP 5: OPTION DATA COLLECTION
# ============================================================

for expiry in expiries:

    expiry_dt = datetime.strptime(expiry, "%Y-%m-%d")   # expiry string ko date me convert
    T = (expiry_dt - today).days / 365   # time to maturity (years me)
    
    if 0 < T <= 60/365:   # sirf next 2 month ka data use karte hain
        
        opt = ticker.option_chain(expiry)   # option chain load
        
        calls = opt.calls.copy()   # call options
        puts = opt.puts.copy()     # put options
        
        calls["type"] = "call"   # identify option type
        puts["type"] = "put"

        calls["expiry"] = expiry_dt   # expiry attach
        puts["expiry"] = expiry_dt

        calls["T"] = T   # maturity time attach
        puts["T"] = T

        all_data.append(pd.concat([calls, puts]))   # call + put combine


# ============================================================
# ✅ STEP 6: COMBINE
# ============================================================

df = pd.concat(all_data)   # sab expiries ka ek dataset


# ============================================================
# ✅ STEP 7: REQUIRED COLUMNS
# ============================================================

df = df[["type","strike","lastPrice","impliedVolatility","volume","expiry","T"]]
# ye hi Heston ke inputs hain → strike (K), price, time, IV etc.


# ============================================================
# ✅ STEP 8: CLEAN DATA
# ============================================================

df = df.dropna()   # missing values hatao
df = df[df["lastPrice"] > 0]   # invalid price hatao
df = df[df["impliedVolatility"] > 0]   # bad IV hatao
df = df[df["volume"] > 10]   # low liquidity data hatao


# ============================================================
# ✅ STEP 9: RISK-FREE RATE
# ============================================================

r = 0.05   # approx risk-free rate (5%) → option discounting ke liye use hota hai


# ============================================================
# ✅ STEP 10: LOG-MONEYNESS
# ============================================================

df["log_moneyness"] = np.log(df["strike"] / spot)  
# log(K/S0) → strike ka relative distance, Heston me important concept


# ============================================================
# ✅ STEP 11: ATM FILTER
# ============================================================

df = df[(df["strike"] > 0.8*spot) & (df["strike"] < 1.2*spot)]  
# ATM (near spot) data use karte hain because deep ITM/OTM noisy hota hai


# ============================================================
# ✅ STEP 12: SORT DATA
# ============================================================

df = df.sort_values(by=["expiry","strike"])   # structured data → calibration stable hota hai


# ============================================================
# ✅ STEP 13: HESTON CHARACTERISTIC FUNCTION
# ============================================================

def heston_cf(u, S0, T, r, kappa, theta, sigma, rho, v0):

    i = complex(0,1)   # imaginary number (Fourier math ke liye)
    x = np.log(S0)     # log spot

    d = np.sqrt((rho*sigma*i*u - kappa)**2 + sigma**2*(i*u + u**2))
    g = (kappa - rho*sigma*i*u - d)/(kappa - rho*sigma*i*u + d)

    C = r*i*u*T + (kappa*theta/sigma**2)*((kappa - rho*sigma*i*u - d)*T - 2*np.log((1 - g*np.exp(-d*T))/(1 - g)))
    D = ((kappa - rho*sigma*i*u - d)/sigma**2)*(1 - np.exp(-d*T))/(1 - g*np.exp(-d*T))

    return np.exp(C + D*v0 + i*u*x)

# ye Heston model ka core math hai → isse probability distribution milti hai


# ============================================================
# ✅ STEP 14: HESTON PRICE FUNCTION
# ============================================================

def heston_price(S0, K, T, r, params):

    kappa, theta, sigma, rho, v0 = params

    def integrand(u):
        i = complex(0,1)
        cf = heston_cf(u - i, S0, T, r, kappa, theta, sigma, rho, v0)
        return np.real(np.exp(-i*u*np.log(K))*cf/(i*u))

    integral = 0
    for j in range(1,100):
        u = j*0.1
        integral += integrand(u)*0.1   # numerical integration

    return S0/2 + integral/np.pi - K*np.exp(-r*T)/2

# ye function option price calculate karta hai (model price)


# ============================================================
# ✅ STEP 15: ERROR FUNCTION
# ============================================================

def error_function(params):

    error = 0

    for i in range(min(50,len(df))):   # speed ke liye few samples
        
        K = df.iloc[i]["strike"]
        T = df.iloc[i]["T"]
        market = df.iloc[i]["lastPrice"]

        model = heston_price(spot, K, T, r, params)

        error += (model - market)**2   # square error

    return error

# calibration ka goal = error minimize karna


# ============================================================
# ✅ STEP 16: OPTIMIZATION
# ============================================================

initial = [1.0, 0.04, 0.3, -0.7, 0.04]   # initial guess parameters

result = minimize(error_function, initial, method='Nelder-Mead')

kappa, theta, sigma, rho, v0 = result.x

print("\nHeston Parameters:")
print("kappa:", kappa)
print("theta:", theta)
print("sigma:", sigma)
print("rho:", rho)
print("v0:", v0)

# ye parameters volatility dynamics represent karte hain


# ============================================================
# ✅ STEP 17: MODEL VS MARKET
# ============================================================

df["model_price"] = df.apply(
    lambda row: heston_price(spot, row["strike"], row["T"], r, result.x),
    axis=1
)

print(df[["strike","lastPrice","model_price"]].head())

# compare karte hain → model vs real price


# ============================================================
# ✅ FINAL OUTPUT
# ============================================================

print("\n✅ Heston Model Ready ✅")

Spot: 312.510009765625

Heston Parameters:
kappa: -286.00319373675757
theta: -14.077177590293417
sigma: -240.12324386315254
rho: -1.070742014326528
v0: 59.2541579477932
   strike  lastPrice  model_price
3   265.0       0.05    34.789247
5   275.0       0.05    27.557411
8   290.0       0.05    17.681234
9   295.0       0.06    14.545068
9   300.0      12.50    11.451582

✅ Heston Model Ready ✅


In [1]:
# ============================================================
# ✅ STEP 1: LIBRARIES IMPORT
# ============================================================

import yfinance as yf              # real market data ke liye
import pandas as pd               # data handling
import numpy as np                # math operations
from datetime import datetime     # time calculation
from scipy.optimize import minimize   # calibration
from scipy.integrate import quad      # accurate integration


# ============================================================
# ✅ STEP 2: ASSET SELECT
# ============================================================

ticker = yf.Ticker("AAPL")   # liquid stock (options data reliable)


# ============================================================
# ✅ STEP 3: SPOT PRICE
# ============================================================

spot = ticker.history(period="1d")["Close"].iloc[-1]  
# latest underlying price (S0)

print("Spot Price:", spot)


# ============================================================
# ✅ STEP 4: FETCH EXPIRIES
# ============================================================

expiries = ticker.options   # available expiries

if len(expiries) == 0:
    raise Exception("No options data found")


today = datetime.today()
all_data = []


# ============================================================
# ✅ STEP 5: OPTION DATA COLLECTION (CALLS ONLY)
# ============================================================

for expiry in expiries:

    expiry_dt = datetime.strptime(expiry, "%Y-%m-%d")
    T = (expiry_dt - today).days / 365   # time to maturity (years)

    if 0 < T <= 60/365:   # sirf near expiries (2 months)

        opt = ticker.option_chain(expiry)
        calls = opt.calls.copy()

        calls["expiry"] = expiry_dt
        calls["T"] = T

        all_data.append(calls)


# ============================================================
# ✅ STEP 6: COMBINE DATA
# ============================================================

df = pd.concat(all_data)


# ============================================================
# ✅ STEP 7: REQUIRED COLUMNS
# ============================================================

df = df[["strike", "lastPrice", "impliedVolatility", "volume", "T"]]


# ============================================================
# ✅ STEP 8: CLEAN DATA
# ============================================================

df = df.dropna()                        # missing hatao
df = df[df["lastPrice"] > 0]            # invalid price hatao
df = df[df["impliedVolatility"] > 0]    # invalid IV hatao
df = df[df["volume"] > 10]              # low liquidity hatao
df = df[df["impliedVolatility"] < 1.0]  # extreme IV hatao


# ============================================================
# ✅ STEP 9: ATM FILTER
# ============================================================

df = df[(df["strike"] > 0.8*spot) & (df["strike"] < 1.2*spot)]


# ============================================================
# ✅ STEP 10: SORT DATA
# ============================================================

df = df.sort_values(by=["T", "strike"])


# ============================================================
# ✅ STEP 11: RISK-FREE RATE
# ============================================================

r = 0.05   # approx 5%


# ============================================================
# ✅ STEP 12: HESTON CHARACTERISTIC FUNCTION
# ============================================================

def heston_cf(u, S0, T, r, kappa, theta, sigma, rho, v0):

    i = complex(0,1)
    x = np.log(S0)#(Genearlly heston model ln(s) ko model instead of stock price)

    d = np.sqrt((rho*sigma*i*u - kappa)**2 + sigma**2*(i*u + u**2))
    g = (kappa - rho*sigma*i*u - d)/(kappa - rho*sigma*i*u + d)

    C = r*i*u*T + (kappa*theta/sigma**2)*(
        (kappa - rho*sigma*i*u - d)*T 
        - 2*np.log((1 - g*np.exp(-d*T))/(1 - g))
    )

    D = ((kappa - rho*sigma*i*u - d)/sigma**2)*(
        (1 - np.exp(-d*T))/(1 - g*np.exp(-d*T))
    )

    return np.exp(C + D*v0 + i*u*x)


# ============================================================
# ✅ STEP 13: HESTON PRICE FUNCTION (P1, P2 METHOD)
# ============================================================

def heston_price(S0, K, T, r, params):

    kappa, theta, sigma, rho, v0 = params
    i = complex(0,1)

    # P1 integral
    def integrand_P1(u):
        cf = heston_cf(u - i, S0, T, r, kappa, theta, sigma, rho, v0)
        return np.real(np.exp(-i*u*np.log(K)) * cf / (i*u*S0*np.exp(r*T)))

    # P2 integral
    def integrand_P2(u):
        cf = heston_cf(u, S0, T, r, kappa, theta, sigma, rho, v0)
        return np.real(np.exp(-i*u*np.log(K)) * cf / (i*u))

    # numerical integration (accurate)
    P1 = 0.5 + (1/np.pi) * quad(integrand_P1, 0, 100)[0]
    P2 = 0.5 + (1/np.pi) * quad(integrand_P2, 0, 100)[0]

    # final call price
    return S0 * P1 - K * np.exp(-r*T) * P2


# ============================================================
# ✅ STEP 14: ERROR FUNCTION (RELATIVE ERROR)
# ============================================================

def error_function(params):

    error = 0
    for i in range(min(50, len(df))):  # speed optimization
        
        K = df.iloc[i]["strike"]
        T = df.iloc[i]["T"]
        market = df.iloc[i]["lastPrice"]

        model = heston_price(spot, K, T, r, params)

        error += ((model - market)/market)**2   # relative error

    return error


# ============================================================
# ✅ STEP 15: OPTIMIZATION WITH CONSTRAINTS
# ============================================================

initial = [1.0, 0.04, 0.3, -0.5, 0.04]

bounds = [
    (0.01, 10),    # kappa > 0
    (0.001, 1),    # theta > 0
    (0.01, 2),     # sigma > 0
    (-0.99, 0.99), # rho
    (0.001, 1)     # v0 > 0
]

result = minimize(error_function, initial, method='L-BFGS-B', bounds=bounds)

kappa, theta, sigma, rho, v0 = result.x


# ============================================================
# ✅ STEP 16: PRINT PARAMETERS
# ============================================================

print("\n✅ Heston Parameters:")
print("kappa:", kappa)
print("theta:", theta)
print("sigma:", sigma)
print("rho:", rho)
print("v0:", v0)


# ============================================================
# ✅ STEP 17: MODEL VS MARKET COMPARISON
# ============================================================

df["model_price"] = df.apply(
    lambda row: heston_price(spot, row["strike"], row["T"], r, result.x),
    axis=1
)

print("\nSample Comparison:")
print(df[["strike", "lastPrice", "model_price"]].head())


# ============================================================
# ✅ FINAL
# ============================================================

print("\n🎯 Heston Model Calibration Completed Successfully!")

Spot Price: 312.510009765625

✅ Heston Parameters:
kappa: 0.01
theta: 0.4755958844972732
sigma: 2.0
rho: -0.9576655924537931
v0: 0.13802820691594017

Sample Comparison:
    strike  lastPrice  model_price
9    300.0      12.50    12.955889
10   305.0       7.20     8.660586
11   310.0       3.77     4.970317
12   315.0       1.20     2.224692
13   320.0       0.27     0.701903

🎯 Heston Model Calibration Completed Successfully!
